### KNN Regression

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV  # Hyperparameter tuning

from sklearn.preprocessing import MinMaxScaler, StandardScaler  #Feature Scaling

from sklearn.neighbors import KNeighborsRegressor

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
df = pd.read_csv('/content/Sales_data.csv')
df

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.300,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.920,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.500,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.200,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.930,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052
...,...,...,...,...,...,...,...,...,...,...,...,...
8518,FDF22,6.865,Low Fat,0.056783,Snack Foods,214.5218,OUT013,1987,High,Tier 3,Supermarket Type1,2778.3834
8519,FDS36,8.380,Regular,0.046982,Baking Goods,108.1570,OUT045,2002,NaN,Tier 2,Supermarket Type1,549.2850
8520,NCJ29,10.600,Low Fat,0.035186,Health and Hygiene,85.1224,OUT035,2004,Small,Tier 2,Supermarket Type1,1193.1136
8521,FDN46,7.210,Regular,0.145221,Snack Foods,103.1332,OUT018,2009,Medium,Tier 3,Supermarket Type2,1845.5976


In [ ]:
s = "DataLearnm"
s[::-1]

'mnraeLataD'

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8523 entries, 0 to 8522
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Item_Identifier            8523 non-null   object 
 1   Item_Weight                7060 non-null   float64
 2   Item_Fat_Content           8523 non-null   object 
 3   Item_Visibility            8523 non-null   float64
 4   Item_Type                  8523 non-null   object 
 5   Item_MRP                   8523 non-null   float64
 6   Outlet_Identifier          8523 non-null   object 
 7   Outlet_Establishment_Year  8523 non-null   int64  
 8   Outlet_Size                6113 non-null   object 
 9   Outlet_Location_Type       8523 non-null   object 
 10  Outlet_Type                8523 non-null   object 
 11  Item_Outlet_Sales          8523 non-null   float64
dtypes: float64(4), int64(1), object(7)
memory usage: 799.2+ KB


In [4]:
df.isna().sum()

,0
Item_Identifier,0
Item_Weight,1463
Item_Fat_Content,0
Item_Visibility,0
Item_Type,0
Item_MRP,0
Outlet_Identifier,0
Outlet_Establishment_Year,0
Outlet_Size,2410
Outlet_Location_Type,0


In [5]:
len(df['Item_Identifier'].unique())

1559

In [6]:
df['Item_Identifier'].value_counts()

,count
Item_Identifier,
FDW13,10
FDG33,10
FDX31,9
FDT07,9
NCY18,9
...,...
FDO33,1
FDK57,1
FDT35,1


#### 1. item weight

In [7]:
df['Item_Weight'].fillna(df['Item_Weight'].median(), inplace=True)

/tmp/ipykernel_193/1915762154.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Item_Weight'].fillna(df['Item_Weight'].median(), inplace=True)


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8523 entries, 0 to 8522
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Item_Identifier            8523 non-null   object 
 1   Item_Weight                8523 non-null   float64
 2   Item_Fat_Content           8523 non-null   object 
 3   Item_Visibility            8523 non-null   float64
 4   Item_Type                  8523 non-null   object 
 5   Item_MRP                   8523 non-null   float64
 6   Outlet_Identifier          8523 non-null   object 
 7   Outlet_Establishment_Year  8523 non-null   int64  
 8   Outlet_Size                6113 non-null   object 
 9   Outlet_Location_Type       8523 non-null   object 
 10  Outlet_Type                8523 non-null   object 
 11  Item_Outlet_Sales          8523 non-null   float64
dtypes: float64(4), int64(1), object(7)
memory usage: 799.2+ KB


#### 2. Item_Fat_Content

In [9]:
df['Item_Fat_Content'].value_counts()

,count
Item_Fat_Content,
Low Fat,5089
Regular,2889
LF,316
reg,117
low fat,112


In [10]:
df['Item_Fat_Content'].replace({'Low Fat':0, 'Regular':1, 'LF':0, 'reg':1, 'low fat':0}, inplace=True)

/tmp/ipykernel_193/2930925291.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Item_Fat_Content'].replace({'Low Fat':0, 'Regular':1, 'LF':0, 'reg':1, 'low fat':0}, inplace=True)
/tmp/ipykernel_193/2930925291.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Item_Fat_Content'].replace({'

In [11]:
df['Item_Fat_Content'].value_counts()

,count
Item_Fat_Content,
0,5517
1,3006


#### Item_Type

In [12]:
df['Item_Type'].value_counts()

,count
Item_Type,
Fruits and Vegetables,1232
Snack Foods,1200
Household,910
Frozen Foods,856
Dairy,682
Canned,649
Baking Goods,648
Health and Hygiene,520
Soft Drinks,445


In [13]:
Item_Type_df = pd.get_dummies(df['Item_Type'], prefix='Item_Type')
Item_Type_df

,Item_Type_Baking Goods,Item_Type_Breads,Item_Type_Breakfast,Item_Type_Canned,Item_Type_Dairy,Item_Type_Frozen Foods,Item_Type_Fruits and Vegetables,Item_Type_Hard Drinks,Item_Type_Health and Hygiene,Item_Type_Household,Item_Type_Meat,Item_Type_Others,Item_Type_Seafood,Item_Type_Snack Foods,Item_Type_Soft Drinks,Item_Type_Starchy Foods
0,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
2,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False
3,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8518,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
8519,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
8520,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False
8521,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False


#### Outlet_Identifier

In [14]:
df['Outlet_Identifier'].value_counts()

,count
Outlet_Identifier,
OUT027,935
OUT013,932
OUT035,930
OUT049,930
OUT046,930
OUT045,929
OUT018,928
OUT017,926
OUT010,555


In [15]:
Outlet_Identifier_df = pd.get_dummies(df['Outlet_Identifier'], prefix='Outlet_Identifier')

In [16]:
Outlet_Identifier_df

,Outlet_Identifier_OUT010,Outlet_Identifier_OUT013,Outlet_Identifier_OUT017,Outlet_Identifier_OUT018,Outlet_Identifier_OUT019,Outlet_Identifier_OUT027,Outlet_Identifier_OUT035,Outlet_Identifier_OUT045,Outlet_Identifier_OUT046,Outlet_Identifier_OUT049
0,False,False,False,False,False,False,False,False,False,True
1,False,False,False,True,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,True
3,True,False,False,False,False,False,False,False,False,False
4,False,True,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...
8518,False,True,False,False,False,False,False,False,False,False
8519,False,False,False,False,False,False,False,True,False,False
8520,False,False,False,False,False,False,True,False,False,False
8521,False,False,False,True,False,False,False,False,False,False


#### Outlet_Size

In [17]:
df['Outlet_Size'].value_counts()

,count
Outlet_Size,
Medium,2793
Small,2388
High,932


In [18]:
df['Outlet_Size'].replace({'Small':0,"Medium":1, "High":2}, inplace=True)

/tmp/ipykernel_193/2321233329.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Outlet_Size'].replace({'Small':0,"Medium":1, "High":2}, inplace=True)
/tmp/ipykernel_193/2321233329.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Outlet_Size'].replace({'Small':0,"Medium":1, "High":2}, in

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8523 entries, 0 to 8522
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Item_Identifier            8523 non-null   object 
 1   Item_Weight                8523 non-null   float64
 2   Item_Fat_Content           8523 non-null   int64  
 3   Item_Visibility            8523 non-null   float64
 4   Item_Type                  8523 non-null   object 
 5   Item_MRP                   8523 non-null   float64
 6   Outlet_Identifier          8523 non-null   object 
 7   Outlet_Establishment_Year  8523 non-null   int64  
 8   Outlet_Size                6113 non-null   float64
 9   Outlet_Location_Type       8523 non-null   object 
 10  Outlet_Type                8523 non-null   object 
 11  Item_Outlet_Sales          8523 non-null   float64
dtypes: float64(5), int64(2), object(5)
memory usage: 799.2+ KB


In [21]:
df['Outlet_Size'].value_counts()

,count
Outlet_Size,
1.0,2793
0.0,2388
2.0,932


In [22]:
df['Outlet_Size'].fillna(1, inplace=True)

#### Outlet_Location_Type

In [23]:
df['Outlet_Location_Type'].value_counts()

,count
Outlet_Location_Type,
Tier 3,3350
Tier 2,2785
Tier 1,2388


In [24]:
df['Outlet_Location_Type'].replace({'Tier 1':0, 'Tier 2':1, 'Tier 3':2}, inplace=True)

/tmp/ipykernel_193/4110967090.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Outlet_Location_Type'].replace({'Tier 1':0, 'Tier 2':1, 'Tier 3':2}, inplace=True)
/tmp/ipykernel_193/4110967090.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Outlet_Location_Type'].replace({'Tier 1':0, '

#### Outlet_Type

In [25]:
df['Outlet_Type'].value_counts()

,count
Outlet_Type,
Supermarket Type1,5577
Grocery Store,1083
Supermarket Type3,935
Supermarket Type2,928


In [26]:
Outlet_Type_df = pd.get_dummies(df['Outlet_Type'], prefix='Outlet_Type',
                                drop_first=True)

In [27]:
Outlet_Type_df

,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3
0,True,False,False
1,False,True,False
2,True,False,False
3,False,False,False
4,True,False,False
...,...,...,...
8518,True,False,False
8519,True,False,False
8520,True,False,False
8521,False,True,False


In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8523 entries, 0 to 8522
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Item_Identifier            8523 non-null   object 
 1   Item_Weight                8523 non-null   float64
 2   Item_Fat_Content           8523 non-null   int64  
 3   Item_Visibility            8523 non-null   float64
 4   Item_Type                  8523 non-null   object 
 5   Item_MRP                   8523 non-null   float64
 6   Outlet_Identifier          8523 non-null   object 
 7   Outlet_Establishment_Year  8523 non-null   int64  
 8   Outlet_Size                8523 non-null   float64
 9   Outlet_Location_Type       8523 non-null   int64  
 10  Outlet_Type                8523 non-null   object 
 11  Item_Outlet_Sales          8523 non-null   float64
dtypes: float64(5), int64(3), object(4)
memory usage: 799.2+ KB


In [29]:
df.drop(['Item_Identifier','Item_Type','Outlet_Identifier', 'Outlet_Type'], axis=1, inplace=True)

In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8523 entries, 0 to 8522
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Item_Weight                8523 non-null   float64
 1   Item_Fat_Content           8523 non-null   int64  
 2   Item_Visibility            8523 non-null   float64
 3   Item_MRP                   8523 non-null   float64
 4   Outlet_Establishment_Year  8523 non-null   int64  
 5   Outlet_Size                8523 non-null   float64
 6   Outlet_Location_Type       8523 non-null   int64  
 7   Item_Outlet_Sales          8523 non-null   float64
dtypes: float64(5), int64(3)
memory usage: 532.8 KB


In [31]:
df_list = [df,Item_Type_df ,Outlet_Identifier_df, Outlet_Type_df]
df = pd.concat(df_list, axis=1)
df

,Item_Weight,Item_Fat_Content,Item_Visibility,Item_MRP,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Item_Outlet_Sales,Item_Type_Baking Goods,Item_Type_Breads,...,Outlet_Identifier_OUT018,Outlet_Identifier_OUT019,Outlet_Identifier_OUT027,Outlet_Identifier_OUT035,Outlet_Identifier_OUT045,Outlet_Identifier_OUT046,Outlet_Identifier_OUT049,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3
0,9.300,0,0.016047,249.8092,1999,1.0,0,3735.1380,False,False,...,False,False,False,False,False,False,True,True,False,False
1,5.920,1,0.019278,48.2692,2009,1.0,2,443.4228,False,False,...,True,False,False,False,False,False,False,False,True,False
2,17.500,0,0.016760,141.6180,1999,1.0,0,2097.2700,False,False,...,False,False,False,False,False,False,True,True,False,False
3,19.200,1,0.000000,182.0950,1998,1.0,2,732.3800,False,False,...,False,False,False,False,False,False,False,False,False,False
4,8.930,0,0.000000,53.8614,1987,2.0,2,994.7052,False,False,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8518,6.865,0,0.056783,214.5218,1987,2.0,2,2778.3834,False,False,...,False,False,False,False,False,False,False,True,False,False
8519,8.380,1,0.046982,108.1570,2002,1.0,1,549.2850,True,False,...,False,False,False,False,True,False,False,True,False,False
8520,10.600,0,0.035186,85.1224,2004,0.0,1,1193.1136,False,False,...,False,False,False,True,False,False,False,True,False,False
8521,7.210,1,0.145221,103.1332,2009,1.0,2,1845.5976,False,False,...,True,False,False,False,False,False,False,False,True,False


In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8523 entries, 0 to 8522
Data columns (total 37 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Item_Weight                      8523 non-null   float64
 1   Item_Fat_Content                 8523 non-null   int64  
 2   Item_Visibility                  8523 non-null   float64
 3   Item_MRP                         8523 non-null   float64
 4   Outlet_Establishment_Year        8523 non-null   int64  
 5   Outlet_Size                      8523 non-null   float64
 6   Outlet_Location_Type             8523 non-null   int64  
 7   Item_Outlet_Sales                8523 non-null   float64
 8   Item_Type_Baking Goods           8523 non-null   bool   
 9   Item_Type_Breads                 8523 non-null   bool   
 10  Item_Type_Breakfast              8523 non-null   bool   
 11  Item_Type_Canned                 8523 non-null   bool   
 12  Item_Type_Dairy     

In [33]:
X = df.drop('Item_Outlet_Sales', axis=1)
y = df['Item_Outlet_Sales']

In [ ]:
df

,Item_Weight,Item_Fat_Content,Item_Visibility,Item_MRP,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Item_Outlet_Sales,Item_Type_Baking Goods,Item_Type_Breads,...,Outlet_Identifier_OUT018,Outlet_Identifier_OUT019,Outlet_Identifier_OUT027,Outlet_Identifier_OUT035,Outlet_Identifier_OUT045,Outlet_Identifier_OUT046,Outlet_Identifier_OUT049,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3
0,9.300,0,0.016047,249.8092,1999,1.0,0,3735.1380,False,False,...,False,False,False,False,False,False,True,True,False,False
1,5.920,1,0.019278,48.2692,2009,1.0,2,443.4228,False,False,...,True,False,False,False,False,False,False,False,True,False
2,17.500,0,0.016760,141.6180,1999,1.0,0,2097.2700,False,False,...,False,False,False,False,False,False,True,True,False,False
3,19.200,1,0.000000,182.0950,1998,1.0,2,732.3800,False,False,...,False,False,False,False,False,False,False,False,False,False
4,8.930,0,0.000000,53.8614,1987,2.0,2,994.7052,False,False,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8518,6.865,0,0.056783,214.5218,1987,2.0,2,2778.3834,False,False,...,False,False,False,False,False,False,False,True,False,False
8519,8.380,1,0.046982,108.1570,2002,1.0,1,549.2850,True,False,...,False,False,False,False,True,False,False,True,False,False
8520,10.600,0,0.035186,85.1224,2004,0.0,1,1193.1136,False,False,...,False,False,False,True,False,False,False,True,False,False
8521,7.210,1,0.145221,103.1332,2009,1.0,2,1845.5976,False,False,...,True,False,False,False,False,False,False,False,True,False


In [34]:
X

,Item_Weight,Item_Fat_Content,Item_Visibility,Item_MRP,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Item_Type_Baking Goods,Item_Type_Breads,Item_Type_Breakfast,...,Outlet_Identifier_OUT018,Outlet_Identifier_OUT019,Outlet_Identifier_OUT027,Outlet_Identifier_OUT035,Outlet_Identifier_OUT045,Outlet_Identifier_OUT046,Outlet_Identifier_OUT049,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3
0,9.300,0,0.016047,249.8092,1999,1.0,0,False,False,False,...,False,False,False,False,False,False,True,True,False,False
1,5.920,1,0.019278,48.2692,2009,1.0,2,False,False,False,...,True,False,False,False,False,False,False,False,True,False
2,17.500,0,0.016760,141.6180,1999,1.0,0,False,False,False,...,False,False,False,False,False,False,True,True,False,False
3,19.200,1,0.000000,182.0950,1998,1.0,2,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,8.930,0,0.000000,53.8614,1987,2.0,2,False,False,False,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8518,6.865,0,0.056783,214.5218,1987,2.0,2,False,False,False,...,False,False,False,False,False,False,False,True,False,False
8519,8.380,1,0.046982,108.1570,2002,1.0,1,True,False,False,...,False,False,False,False,True,False,False,True,False,False
8520,10.600,0,0.035186,85.1224,2004,0.0,1,False,False,False,...,False,False,False,True,False,False,False,True,False,False
8521,7.210,1,0.145221,103.1332,2009,1.0,2,False,False,False,...,True,False,False,False,False,False,False,False,True,False


In [ ]:
# norm_scaler = MinMaxScaler()
# X_nor = norm_scaler.fit_transform(X)
# # X_nor
# X_nor_df = pd.DataFrame(X_nor, columns=X.columns)
# X_nor_df

### Train Test Split

In [35]:
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                            test_size=0.15, random_state=42)

In [36]:
X_train

,Item_Weight,Item_Fat_Content,Item_Visibility,Item_MRP,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Item_Type_Baking Goods,Item_Type_Breads,Item_Type_Breakfast,...,Outlet_Identifier_OUT018,Outlet_Identifier_OUT019,Outlet_Identifier_OUT027,Outlet_Identifier_OUT035,Outlet_Identifier_OUT045,Outlet_Identifier_OUT046,Outlet_Identifier_OUT049,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3
2221,7.100,0,0.109991,172.9080,2004,0.0,1,False,False,False,...,False,False,False,True,False,False,False,True,False,False
7535,10.650,1,0.085268,229.7668,1999,1.0,0,False,False,False,...,False,False,False,False,False,False,True,True,False,False
4362,9.310,0,0.037955,62.1510,1997,0.0,0,False,False,False,...,False,False,False,False,False,True,False,True,False,False
1665,6.780,1,0.000000,227.5694,1999,1.0,0,False,False,False,...,False,False,False,False,False,False,True,True,False,False
6678,10.100,0,0.093862,115.9492,1998,1.0,2,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5734,9.395,1,0.286345,139.1838,1998,1.0,2,False,False,False,...,False,False,False,False,False,False,False,False,False,False
5191,15.600,0,0.117575,75.6670,2007,1.0,1,False,False,False,...,False,False,False,False,False,False,False,True,False,False
5390,17.600,0,0.018944,237.3590,2002,1.0,1,False,False,False,...,False,False,False,False,True,False,False,True,False,False
860,20.350,0,0.054363,117.9466,2007,1.0,1,False,False,False,...,False,False,False,False,False,False,False,True,False,False


In [37]:
X_test

,Item_Weight,Item_Fat_Content,Item_Visibility,Item_MRP,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Item_Type_Baking Goods,Item_Type_Breads,Item_Type_Breakfast,...,Outlet_Identifier_OUT018,Outlet_Identifier_OUT019,Outlet_Identifier_OUT027,Outlet_Identifier_OUT035,Outlet_Identifier_OUT045,Outlet_Identifier_OUT046,Outlet_Identifier_OUT049,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3
7503,14.300,0,0.026300,79.4302,1987,2.0,2,False,False,False,...,False,False,False,False,False,False,False,True,False,False
2957,7.930,0,0.071136,42.7086,1997,0.0,0,False,False,False,...,False,False,False,False,False,True,False,True,False,False
7031,14.500,1,0.041313,42.0454,1999,1.0,0,False,False,False,...,False,False,False,False,False,False,True,True,False,False
1084,12.600,1,0.044767,173.7054,1985,1.0,2,False,False,False,...,False,False,True,False,False,False,False,False,False,True
856,10.195,1,0.012456,197.5110,2004,0.0,1,False,False,False,...,False,False,False,True,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1349,11.650,0,0.039981,227.3694,1999,1.0,0,False,False,False,...,False,False,False,False,False,False,True,True,False,False
3018,12.600,1,0.148737,155.1314,1985,0.0,0,False,False,False,...,False,True,False,False,False,False,False,False,False,False
3992,16.000,1,0.173463,157.6972,2009,1.0,2,False,False,False,...,True,False,False,False,False,False,False,False,True,False
8465,16.000,1,0.106969,180.5634,2002,1.0,1,True,False,False,...,False,False,False,False,True,False,False,True,False,False


### Train Model

In [38]:
knn_reg = KNeighborsRegressor(n_neighbors=11, p=2)
knn_reg.fit(X_train, y_train)

KNeighborsRegressor(n_neighbors=11)

In [39]:
print(type(X_test))

<class 'pandas.core.frame.DataFrame'>


In [40]:
X_test = X_test.fillna(0)

In [42]:
X_test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1279 entries, 7503 to 6685
Data columns (total 36 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Item_Weight                      1279 non-null   float64
 1   Item_Fat_Content                 1279 non-null   int64  
 2   Item_Visibility                  1279 non-null   float64
 3   Item_MRP                         1279 non-null   float64
 4   Outlet_Establishment_Year        1279 non-null   int64  
 5   Outlet_Size                      1279 non-null   float64
 6   Outlet_Location_Type             1279 non-null   int64  
 7   Item_Type_Baking Goods           1279 non-null   bool   
 8   Item_Type_Breads                 1279 non-null   bool   
 9   Item_Type_Breakfast              1279 non-null   bool   
 10  Item_Type_Canned                 1279 non-null   bool   
 11  Item_Type_Dairy                  1279 non-null   bool   
 12  Item_Type_Frozen Foods

In [43]:
from sklearn.neighbors import KNeighborsRegressor

knn_reg = KNeighborsRegressor(n_neighbors=5)
knn_reg.fit(X_train, y_train)

KNeighborsRegressor()

In [44]:
X_test.isna().sum()

,0
Item_Weight,0
Item_Fat_Content,0
Item_Visibility,0
Item_MRP,0
Outlet_Establishment_Year,0
Outlet_Size,0
Outlet_Location_Type,0
Item_Type_Baking Goods,0
Item_Type_Breads,0
Item_Type_Breakfast,0


In [45]:
y_pred = knn_reg.predict(X_test)
y_pred

array([ 920.40192,  860.74624,  734.77688, ..., 2546.55184, 2925.5252 ,
        456.87196])

In [46]:
len(y_pred)

1279

In [47]:
y_test[:5]

,Item_Outlet_Sales
7503,1743.0644
2957,356.8688
7031,377.5086
1084,5778.4782
856,2356.9320


In [48]:
y_pred[:5]

array([ 920.40192,  860.74624,  734.77688, 3553.10828, 3860.57472])

### Evaluation

In [49]:
# Mean Squared error
mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error is :", mse)

Mean Squared Error is : 1384543.4751766997


In [50]:
# Root Mean Squared error
rmse = np.sqrt(mse)
print("Root Mean Squared Error is :", rmse)

Root Mean Squared Error is : 1176.6662547964481


In [51]:
# Mean Absolute Error
mae = mean_absolute_error(y_test, y_pred)
print("Mean Absolute Error is :", mae)

Mean Absolute Error is : 835.0698894448788


In [52]:
# Accuracy score
accuracy = r2_score(y_test, y_pred)
print("Accuarcy of knn model is :", accuracy)

Accuarcy of knn model is : 0.5078847313117361


### Hyperparameter Tuning

#### Grid SearchCV

In [71]:
k = np.arange(2,20)
p = [1,2]
hyp = {'n_neighbors': k, "p":p}

In [72]:
knn = KNeighborsRegressor()
best_knn_model = GridSearchCV(knn,hyp,cv=5)
best_knn_model.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=KNeighborsRegressor(),
             param_grid={'n_neighbors': array([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18,
       19]),
                         'p': [1, 2]})

In [73]:
best_knn_model.best_params_

{'n_neighbors': np.int64(13), 'p': 1}

#### Randomized Search CV

In [56]:
knn = KNeighborsRegressor()
best_knn_model = RandomizedSearchCV(knn,hyp,cv=5)
best_knn_model.fit(X_train, y_train)

RandomizedSearchCV(cv=5, estimator=KNeighborsRegressor(),
                   param_distributions={'n_neighbors': array([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14]),
                                        'p': [1, 2]})

In [57]:
best_knn_model.best_params_

{'p': 1, 'n_neighbors': np.int64(5)}

In [58]:
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                            test_size=0.2, random_state=42)

In [68]:
knn_reg1 = KNeighborsRegressor(n_neighbors=14, p=2)
knn_reg1.fit(X_train, y_train)

KNeighborsRegressor(n_neighbors=14)

In [69]:
y_pred1 = knn_reg1.predict(X_test)
y_pred1

array([1557.54398571,  733.61648571,  686.29712857, ...,  762.91168571,
        641.78364286, 1593.78252857])

In [70]:
# Accuracy score
accuracy = r2_score(y_test, y_pred1)
print("Accuarcy of knn model is :", accuracy)

Accuarcy of knn model is : 0.4848222745970401


In [65]:
knn_reg2 = KNeighborsRegressor(n_neighbors=5, p=1)
knn_reg2.fit(X_train, y_train)

KNeighborsRegressor(p=1)

In [66]:
y_pred2 = knn_reg2.predict(X_test)
y_pred2

array([1135.8548 ,  848.36236,  734.77688, ...,  635.30636,  562.20152,
       1337.85852])

In [67]:
# Accuracy score
accuracy = r2_score(y_test, y_pred2)
print("Accuarcy of knn model is :", accuracy)

Accuarcy of knn model is : 0.5193190796776095
